# Colab GPU → Ada Nest worker (reverse-SSH variant)

Turns this Colab runtime into a fleet-reachable inference node — no tailnet needed:

1. Installs **Ollama** on the T4 and serves an OpenAI-compatible API on **loopback** `:11434`.
2. Generates a throwaway SSH keypair here, you send the **public** key to Devin — it lands
   on `colabtun@idc03` with tunnel-only restrictions (`nologin`, `permitopen` only :11434).
3. Opens `ssh -N -R 11434:127.0.0.1:11434 colabtun@idc03` — ollama appears on **idc03's loopback**,
   which the LiteLLM gateway (host-net :4000) can reach directly.

Runtime → change runtime type → **T4 GPU** before starting.

When the Colab session dies, the tunnel drops and LiteLLM falls through to the next
chain entry — nothing breaks.

In [ ]:
# ---- settings -----------------------------------------------------------
MODEL = "gemma3:12b"      # ~8 GB q4 — fits T4 15 GB. Alternatives: qwen3:8b, llama3.1:8b, gemma3:4b
TUNNEL_USER = "colabtun"
TUNNEL_HOST = "157.85.102.125"   # idc03 public IP (ssh :22)
print("target model:", MODEL)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# ---- install ollama (script, tarball fallback, visible errors) ------------
import shutil, subprocess, os

# ollama >= 0.11 ships zstd-packed tarballs; Colab's base image lacks zstd
# and install.sh dies with "This version requires zstd for extraction".
if not shutil.which("zstd"):
    subprocess.run("apt-get install -y zstd", shell=True)

if shutil.which("ollama"):
    print("ollama already installed:", shutil.which("ollama"))
else:
    r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:])
    if not shutil.which("ollama"):
        print("install.sh failed — trying direct tarball…")
        r = subprocess.run(
            "curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst "
            "| tar -x -C /usr/local", shell=True, capture_output=True, text=True)
        print((r.stdout + r.stderr)[-800:])
    print("ollama:", shutil.which("ollama") or "STILL MISSING")
    subprocess.run(["ldconfig"], capture_output=True)

In [ ]:
# ---- generate this session's keypair — send the PUBLIC key to Devin -------
import os, subprocess
os.makedirs("/root/.ssh", exist_ok=True)
if not os.path.exists("/root/.ssh/colab_tun"):
    subprocess.run(["ssh-keygen", "-t", "ed25519", "-f", "/root/.ssh/colab_tun",
                    "-N", "", "-C", "colab-serve-worker"], check=True)
pub = open("/root/.ssh/colab_tun.pub").read().strip()
print("=" * 70)
print("SEND THIS PUBLIC KEY TO DEVIN (safe to share — it's public):")
print()
print(pub)
print()
print("Wait for 'installed on colabtun@idc03' before running the next cell.")
print("=" * 70)

In [ ]:
# ---- open the reverse tunnel (run AFTER Devin confirms the key) ----------
import subprocess, time, os

subprocess.run(["ssh-keyscan", "-T", "8", "-t", "ed25519", TUNNEL_HOST],
               stdout=open("/root/.ssh/known_hosts", "a"),
               stderr=subprocess.DEVNULL)

# PasswordAuthentication=no: without it ssh falls back to password auth when
# the key isn't authorized yet — each failure is logged by sshd and fail2ban
# bans the Colab egress IP (incident 2026-10-12).
cmd = ("while true; do "
       f"ssh -N -T -i /root/.ssh/colab_tun "
       "-o PasswordAuthentication=no -o IdentitiesOnly=yes "
       "-o ServerAliveInterval=30 -o ServerAliveCountMax=3 "
       "-o ExitOnForwardFailure=yes "
       "-R 127.0.0.1:11434:127.0.0.1:11434 "
       f"{TUNNEL_USER}@{TUNNEL_HOST} ; "
       "echo '[tunnel] disconnected, retrying in 15s'; sleep 15; done")

subprocess.Popen(["bash", "-c", cmd],
                 stdout=open("/tmp/tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
print(open("/tmp/tunnel.log").read() or "(tunnel starting…)")
r = subprocess.run(["pgrep", "-af", "ssh -N"], capture_output=True, text=True)
print("tunnel process:", r.stdout.strip() or "NOT RUNNING — check key install")

In [ ]:
# ---- start ollama on loopback (only the tunnel needs it) -----------------
import subprocess, os, time, urllib.request, shutil
OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen([OLLAMA_BIN, "serve"], env=env,
                 stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for i in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())

In [ ]:
# ---- pull the model (first run downloads ~8 GB) --------------------------
!ollama pull $MODEL
!ollama list

In [ ]:
# ---- smoke test (local side) ----------------------------------------------
import json, urllib.request
req = urllib.request.Request(
    "http://127.0.0.1:11434/v1/chat/completions",
    data=json.dumps({"model": MODEL,
                     "messages": [{"role": "user", "content":
                                   "Reply with exactly: colab worker ready"}]}).encode(),
    headers={"Content-Type": "application/json"})
print(json.loads(urllib.request.urlopen(req, timeout=180).read())
        ["choices"][0]["message"]["content"])

In [ ]:
# ---- print the fleet-side registration ------------------------------------
print(f"""
═══════════════════════════════════════════════════════════════════
 Tunnel maps idc03 127.0.0.1:11434 -> this runtime's ollama.

 Register into the LiteLLM gateway (Devin runs this):

   curl -s -X POST http://100.102.134.91:4000/model/new \\
     -H 'Authorization: Bearer $LITELLM_MASTER_KEY' \\
     -H 'Content-Type: application/json' \\
     -d '{{"model_name": "colab-{MODEL.split(':')[0]}",
          "litellm_params": {{"model": "ollama_chat/{MODEL}",
                              "api_base": "http://127.0.0.1:11434"}}}}'

 (api_base is idc03's loopback — host-net litellm reaches it directly.)
═══════════════════════════════════════════════════════════════════
""")

In [ ]:
# ---- keep-alive heartbeat — leave RUNNING to keep the node alive ----------
import time, subprocess, urllib.request
while True:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    tun = subprocess.run(["pgrep", "-f", "ssh -N"], capture_output=True).returncode == 0
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=3)
        ok = "up"
    except Exception:
        ok = "DOWN"
    print(time.strftime("%H:%M:%S"), "| gpu:", gpu,
          "| tunnel:", "UP" if tun else "DOWN", "| ollama:", ok, flush=True)
    time.sleep(300)